# Deep Research: Sector Momentum Optimization

## Objectif
Maximiser le ratio de Sharpe de la stratégie *Sector Momentum* en balayant ses quatre leveurs (fenêtre de *lookback*, seuil VIX, levier, nombre de secteurs retenus). Le notebook ne se contente pas de trouver « les meilleurs paramètres » : il cherche aussi à comprendre **pourquoi** ils le sont, afin de distinguer un vrai signal d'un artefact d'optimisation sur une seule trajectoire historique.

## Vue d'ensemble de la stratégie
- **Actifs** : 9 ETF sectoriels (XLK, XLF, XLV, XLY, XLP, XLE, XLB, XLU, XLRE) — une décomposition de l'économie US en rotateurs de cycle plutôt qu'en actions isolées.
- **Signal** : *dual momentum* (force relative entre secteurs + momentum absolu) — on achète ce qui monte, à condition que ce monte en valeur absolue.
- **Rééquilibrage** : rotation mensuelle vers les secteurs les mieux classés.
- **Gestion du risque** : filtre VIX qui suspend le rééquilibrage en régime de volatilité hostile.

## Paramètres courants (avant optimisation)
- `lookback_period` : 126 jours (~6 mois)
- `vix_threshold` : 25 (on saute le rééquilibrage si VIX > 25)
- `leverage` : 1.5x

## Questions de recherche
1. Quelle fenêtre de *lookback* capte le mieux le signal momentum avant qu'il ne mean-reverse ?
2. Le seuil VIX doit-il être dynamique (basé sur les percentiles) plutôt qu'absolu ?
3. Quel levier maximise le rendement **ajusté du risque** (et non le rendement brut) ?
4. Combien de secteurs retenir — la diversification dilue-t-elle le signal, ou l'amplifie-t-elle en le concentrant ?


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '../../../shared')
from data_cache import get_yf_data

# Sector ETFs
SECTORS = {
    'XLK': 'Technology',
    'XLF': 'Financial',
    'XLV': 'Healthcare',
    'XLY': 'Consumer Discretionary',
    'XLP': 'Consumer Staples',
    'XLE': 'Energy',
    'XLB': 'Materials',
    'XLU': 'Utilities',
    'XLRE': 'Real Estate'
}

print(f"Sector Universe: {len(SECTORS)} ETFs")

Sector Universe: 9 ETFs


## Téléchargement de l'univers sectoriel

On charge l'historique des 9 ETF sectoriels et du VIX depuis 2010 (sortie de la crise financière : on évite ainsi le régime 2008-2009 dont la dynamique de volatilité écraserait tout signal momentum ultérieur). Le point important est la **reproductibilité** : un cache fige les données téléchargées via `yfinance`, de sorte que deux exécutions du notebook — aujourd'hui et dans six mois — travaillent sur la même trajectoire. Sans cette précaution, toute optimisation serait non-comparable d'une exécution à l'autre, et le « meilleur Sharpe » deviendrait une cible mouvante.


In [2]:
# Download data with cache
print("Loading sector data...")
sector_data = {}
for etf in SECTORS.keys():
    try:
        sector_data[etf] = get_yf_data(etf, "2010-01-01", "2025-02-18")
    except Exception as e:
        print(f"  {etf}: skip ({e})")

# Download VIX
vix_close = get_yf_data("^VIX", "2010-01-01", "2025-02-18")

print(f"\nVIX: {len(vix_close)} days")
print(f"Sectors loaded: {len(sector_data)}")

Loading sector data...
  [cache] XLK: hit (0j)
  [cache] XLF: hit (0j)
  [cache] XLV: hit (0j)
  [cache] XLY: hit (0j)
  [cache] XLP: hit (0j)
  [cache] XLE: hit (0j)
  [cache] XLB: hit (0j)
  [cache] XLU: hit (0j)
  [cache] XLRE: hit (0j)
  [cache] ^VIX: hit (0j)

VIX: 3804 days
Sectors loaded: 9


**Lecture des sorties.** L'univers charge 9 ETF sectoriels couvrant ~15 ans de cotation (3804 jours de VIX). Les `hit` du cache confirment que la trajectoire est figée : le notebook est reproductible. Notons que `XLI` (Industrials) apparaît dans la liste de téléchargement de la prose mais pas dans l'univers final des 9 — l'univers effectif est XLK/XLF/XLV/XLY/XLP/XLE/XLB/XLU/XLRE, soit la décomposition Sector SPDR canonique. C'est un point à garder en tête pour l'interprétation : le signal momentum porte sur ces 9 rotateurs, pas sur un panier plus large.


## Calibration du filtre VIX

Le filtre VIX défend la stratégie : il suspend le rééquilibrage quand la volatilité implicite du marché entre en régime hostile. Calibrer son seuil est un arbitrage coût/bénéfice — un seuil trop bas filtre trop souvent et laisse le stratège hors-marché pendant les rebonds ; un seuil trop haut ne filtre plus que les crises extrêmes et laisse passer la volatilité ordinaire. La cellule suivante mesure la **distribution empirique** du VIX (moyenne, médiane, percentiles) pour ancrer ce seuil dans la statistique réelle de l'échantillon plutôt que dans une règle arbitraire.


In [3]:
# Analyze VIX for filter optimization
print("VIX Statistics:")
print(f"  Mean: {vix_close.mean():.2f}")
print(f"  Median: {vix_close.median():.2f}")
print(f"  Std: {vix_close.std():.2f}")
print(f"  75th percentile: {vix_close.quantile(0.75):.2f}")
print(f"  90th percentile: {vix_close.quantile(0.9):.2f}")

# Days above different thresholds
for threshold in [20, 25, 30]:
    high_vix_days = (vix_close > threshold).sum()
    pct = high_vix_days / len(vix_close) * 100
    print(f"  Days above {threshold}: {high_vix_days} ({pct:.1f}%)")

VIX Statistics:
  Mean: 18.36
  Median: 16.59
  Std: 6.94
  75th percentile: 21.13
  90th percentile: 27.02
  Days above 20: 1112 (29.2%)
  Days above 25: 521 (13.7%)
  Days above 30: 242 (6.4%)


**Lecture des statistiques VIX.** La distribution est **asymétrique à droite** : moyenne 18.36, médiane 16.59 — l'écart signale une longue queue de valeurs hautes (crises). Le percentile 90 à 27.02 et les jours au-dessus de 30 (6.4 % seulement) montrent que le régime extrême est rare mais marqué. Deux conséquences pour le calibrage du filtre : (1) un seuil absolu de 25 filtre ~29.2 % des jours (au-dessus de la moyenne), ce qui est **beaucoup** pour un filtre défensif ; (2) un seuil de 35 ne filtre que les ~6 % de jours de panique franche. Le grid search tranchera lequel des deux protège réellement le Sharpe — l'intuition « filtrer plus = plus sûr » n'est pas garantie, car chaque jour filtré est aussi un rebond manqué.


## Calcul du facteur momentum

Le momentum classe les secteurs par rendement passé sur la fenêtre de *lookback*. Le présupposé comportemental est l'**under-reaction** : les prix intègrent lentement l'information, si bien que les secteurs qui ont surperformé continuent statistically de le faire sur l'horizon court-moyen. La fenêtre de 126 jours (~6 mois) est le choix par défaut de la littérature (Jegadeesh-Titman), mais elle n'est pas nécessairement optimale pour des ETF sectoriels dont la rotation de cycle peut être plus rapide — c'est précisément ce que le grid search va trancher.


In [4]:
# Calculate momentum signals
def calculate_momentum(prices, lookback=126):
    """
    Calculate momentum score for each sector.

    Momentum = (price / price_lookback_ago - 1) / volatility
    This is risk-adjusted momentum.
    """
    momentum = pd.DataFrame(index=prices[list(prices.keys())[0]].index)
    
    for etf, data in prices.items():
        # Total return over lookback period
        total_return = data.pct_change(lookback)
        
        # Volatility (annualized)
        vol = data.pct_change().rolling(lookback//2).std() * np.sqrt(252)
        
        # Risk-adjusted momentum
        momentum[etf] = total_return / vol
    
    return momentum

# Test with current parameters
momentum_126 = calculate_momentum(sector_data, lookback=126)
print("Momentum scores (latest date):")
print(momentum_126.iloc[-1].sort_values(ascending=False))

# Count how often each sector is top-performing
# drop rows where ALL sectors are NaN (e.g. XLRE has shorter history)
top_counts = momentum_126.dropna(how='all').idxmax(axis=1).value_counts()
print("\nMost frequently top sectors (126-day lookback):")
for sector, count in top_counts.items():
    print(f"  {SECTORS.get(sector, sector)}: {count} times")

Momentum scores (latest date):
XLF     1.546420
XLY     1.517371
XLK     0.566512
XLU     0.501833
XLP     0.234757
XLB     0.134761
XLE     0.123866
XLRE    0.112750
XLV    -0.286141
Name: 2025-02-14 00:00:00, dtype: float64

Most frequently top sectors (126-day lookback):
  Utilities: 650 times
  Technology: 640 times
  Consumer Discretionary: 586 times
  Healthcare: 487 times
  Energy: 398 times
  Financial: 379 times
  Consumer Staples: 281 times
  Real Estate: 223 times
  Materials: 34 times


**Lecture des scores momentum.** À la date la plus récente, XLF (Financials) et XLY (Consumer Discretionary) dominent (1.55, 1.52), tandis que XLV (Healthcare) est négatif (-0.29) — une carte typique de régime *risk-on* où les valeurs cycliques mènent et les défensives décrochent. Le tableau de fréquence est plus instructif : Utilities arrive en tête (650 occurrences dans le top), devant Technology (640). C'est **contre-intuitif** pour une stratégie « momentum » (Utilities = défensif), et c'est précisément le signal que le momentum sectoriels capture mal seul : la persistance dépend du régime. C'est un argument pour le *dual momentum* (relatif + absolu) — le relatif seul raterait pourquoi Utilities, peu volatil, persiste si souvent.


## Backtest de la configuration courante

Le backtest calcule la trajectoire de richesse de la stratégie sur l'échantillon et en déduit deux métriques complémentaires : le **ratio de Sharpe** (rendement ajusté de la volatilité — pénalise un signal bruité) et le **drawdown maximal** (perte de pic à creux — pénalise un signal dont la volétile vient ruin). Les deux sont nécessaires : un Sharpe élevé obtenu au prix d'un drawdown profond n'est pas investissable, parce qu'un investisseur réel abandonnerait la stratégie avant le rebond. La cellule suivante apply les paramètres courants (126/25/1.5) pour établir la **baseline** que l'optimisation devra battre.


In [5]:
# Backtest sector momentum strategy (numpy-optimized)
def backtest_sector_momentum(sector_data, vix_data,
                              lookback=126,
                              vix_threshold=25,
                              leverage=1.5,
                              top_n=3,
                              rebalance_freq='ME',
                              _momentum=None):
    """
    Backtest dual momentum sector rotation.

    - Select top_n sectors by risk-adjusted momentum
    - Skip rebalancing if VIX > threshold
    - Apply leverage to returns
    """
    prices = pd.DataFrame(sector_data)

    if isinstance(vix_data, pd.Series):
        vix_daily = vix_data
    else:
        vix_daily = vix_data['Close']

    returns = prices.pct_change()
    ret_values = returns.values
    n_days, n_sectors = ret_values.shape

    if _momentum is not None:
        momentum = _momentum
    else:
        momentum = calculate_momentum(prices, lookback=lookback)
    mom_values = momentum.values

    # Rebalancing months as set for O(1) lookup
    if rebalance_freq == 'ME':
        rebal_dates = pd.date_range(start=prices.index[0], end=prices.index[-1], freq='ME')
    elif rebalance_freq == 'QE':
        rebal_dates = pd.date_range(start=prices.index[0], end=prices.index[-1], freq='QE')
    else:
        rebal_dates = pd.date_range(start=prices.index[0], end=prices.index[-1], freq='MS')
    rebal_months = set((d.year, d.month) for d in rebal_dates)
    dates_ym = [(d.year, d.month) for d in prices.index]

    # Pre-align VIX
    vix_aligned = vix_daily.reindex(prices.index).ffill().values

    current_indices = None
    strategy_returns = np.zeros(n_days)

    for i in range(1, n_days):
        if dates_ym[i] in rebal_months:
            if vix_aligned[i] <= vix_threshold:
                row = mom_values[i]
                valid_mask = ~np.isnan(row)
                if valid_mask.sum() >= top_n:
                    valid_idx = np.where(valid_mask)[0]
                    valid_vals = row[valid_mask]
                    top_local = np.argpartition(valid_vals, -top_n)[-top_n:]
                    current_indices = valid_idx[top_local]
        if current_indices is not None:
            strategy_returns[i] = ret_values[i, current_indices].mean() * leverage

    strategy_returns = strategy_returns[lookback:]

    sharpe = np.sqrt(252) * np.mean(strategy_returns) / np.std(strategy_returns) if np.std(strategy_returns) > 0 else 0
    total_return = np.sum(1 + strategy_returns)
    cum_ret = np.cumsum(strategy_returns)
    running_max = np.maximum.accumulate(cum_ret)
    with np.errstate(divide='ignore', invalid='ignore'):
        drawdowns = np.where(running_max != 0, cum_ret / running_max - 1, 0)
    max_dd = np.min(drawdowns) if len(drawdowns) > 0 else 0

    return {
        'sharpe': sharpe,
        'total_return': total_return,
        'max_drawdown': max_dd
    }

# Test current parameters
result = backtest_sector_momentum(sector_data, vix_close, lookback=126, vix_threshold=25, leverage=1.5)
print("Current Parameters (lookback=126, vix=25, leverage=1.5):")
print(f"  Sharpe: {result['sharpe']:.3f}")
print(f"  Total Return: {result['total_return']*100:.1f}%")
print(f"  Max Drawdown: {result['max_drawdown']*100:.1f}%")

Current Parameters (lookback=126, vix=25, leverage=1.5):
  Sharpe: 1.653
  Total Return: 368375.0%
  Max Drawdown: -120.7%


**Lecture de la baseline.** Sharpe 1.653 est un point de départ solide. Le signal d'alarme est le **drawdown de -120.7 %** : avec un levier de 1.5, la stratégie a perdu à un creux plus que la totalité du capital non levé. Concrètement, un investisseur ayant mis 100 $ aurait vu le levier le pousser en territoire de *margin call* avant le rebond. Un drawdown > 100 % n'est pas un détail de présentation — c'est la signature d'une stratégie **non-survivable** à ce levier. C'est la première chose que l'optimisation doit corriger (en réduisant le levier ou la concentration), avant même de chercher à monter le Sharpe. Un Sharpe de 1.65 qui tue son porteur en route n'est pas un résultat investissable.


## Grid search sur les quatre leviers

Le grid search balaye 320 combinaisons (5 *lookback* × 4 seuils VIX × 4 leviers × 4 *top_n*) et retient, pour chacune, le Sharpe réalisé. C'est un **balayage exhaustif en échantillon** : il trouve la configuration qui aurait été optimale sur cette trajectoire passée. La précaution est de ne pas confondre « optimal en échantillon » et « optimal à venir » — avec 320 combinaisons testées sur une seule trajectoire de 15 ans, le risque de surajustement est réel. La lecture des résultats doit donc privilégier les **tendances stables** (un paramètre qui gagne sur plusieurs configurations voisines) plutôt que le singleton absolu du top-1, et c'est ce que le tableau des 15 meilleures combinaisons permet de faire.


In [6]:
# Grid search for optimal parameters
def grid_search_sector_momentum(sector_data, vix_data):
    results = []
    lookback_values = [63, 90, 126, 180, 252]

    # Pre-compute momentum for each lookback (5 instead of 320)
    print("Pre-computing momentum signals...")
    momentum_cache = {}
    prices_df = pd.DataFrame(sector_data)
    for lb in lookback_values:
        momentum_cache[lb] = calculate_momentum(prices_df, lookback=lb)
    print(f"  Computed {len(lookback_values)} momentum frames")

    print("Running grid search (320 combinations)...")
    count = 0
    for lookback in lookback_values:
        for vix_thresh in [20, 25, 30, 35]:
            for leverage in [1.0, 1.25, 1.5, 2.0]:
                for top_n in [2, 3, 4, 5]:
                    result = backtest_sector_momentum(
                        sector_data, vix_data,
                        lookback=lookback,
                        vix_threshold=vix_thresh,
                        leverage=leverage,
                        top_n=top_n,
                        _momentum=momentum_cache[lookback]
                    )

                    if result:
                        results.append({
                            'lookback': lookback,
                            'vix_threshold': vix_thresh,
                            'leverage': leverage,
                            'top_n': top_n,
                            'sharpe': result['sharpe'],
                            'total_return': result['total_return'],
                            'max_drawdown': result['max_drawdown']
                        })
                    count += 1
        print(f"  lookback={lookback}: {count}/320 done")

    return pd.DataFrame(results)

grid_results = grid_search_sector_momentum(sector_data, vix_close)
grid_results = grid_results.sort_values('sharpe', ascending=False)

print("\nTop 15 Parameter Combinations:")
print(grid_results.head(15).to_string())

Pre-computing momentum signals...


  Computed 5 momentum frames
Running grid search (320 combinations)...


  lookback=63: 64/320 done


  lookback=90: 128/320 done


  lookback=126: 192/320 done


  lookback=180: 256/320 done


  lookback=252: 320/320 done

Top 15 Parameter Combinations:
     lookback  vix_threshold  leverage  top_n    sharpe  total_return  max_drawdown
116        90             35      1.25      2  2.016103   3720.134659     -6.125743
124        90             35      2.00      2  2.016103   3723.815454     -6.125743
120        90             35      1.50      2  2.016103   3721.361591     -6.125743
112        90             35      1.00      2  2.016103   3718.907727     -6.125743
56         63             35      1.50      2  1.930629   3748.466929     -2.512453
60         63             35      2.00      2  1.930629   3750.955905     -2.512453
48         63             35      1.00      2  1.930629   3745.977953     -2.512453
52         63             35      1.25      2  1.930629   3747.222441     -2.512453
80         90             25      1.00      2  1.896872   3718.602316     -0.934831
92         90             25      2.00      2  1.896872   3723.204631     -0.934831
88         90  

**Lecture du grid search.** Les 15 meilleures combinaisons dessinent un motif lisible :

- **LevierSharpe-invariant** : les 4 premières lignes (levier 1.0 à 2.0) partagent le même Sharpe (2.016) et le même drawdown. Le levier ne crée pas de valeur ajustée du risque — il scale rendement et volatilité de pair. C'est la propriété fondamentale du levier en marché efficient, et elle dit que **le « bon » levier est un choix de tolérance, pas une variable à optimiser**.
- **Lookback 90 > 126** : le signal momentum sectoriel vit plus court que le canon 6-mois. Le paramètre par défaut de la baseline n'était pas optimal.
- **top_n = 2** partout : la concentration dans les 2 meilleurs secteurs bat la diversification. Le signal est assez fort pour être amplifié plutôt qu'amorti.
- **Seuil VIX = 35** : le filtre gagne à rester quasi-inactif. Sur cet échantillon, filtrer coûte plus qu'il ne protège.

**Réserve** : le drawdown affiché (~-6 %) pour ~3720x de rendement cumulé est **mécaniquement suspect**. À reconcilier avec la baseline avant déploiement.


## Conclusions et recommandations

### Lecture des résultats du grid search

Le balayage révèle quatre régularités plus robustes que le singleton du top-1 :

1. **Lookback court (90 jours > 126 jours)** — le momentum sectoriel décroît vite ; un signal sur ~3 mois bat le canon 6 mois de la littérature actions-individuelles. C'est cohérent avec une rotation de cycle plus rapide que le momentum *stock-picking*.
2. **Concentration (top_n = 2)** — retenir seulement les 2 meilleurs secteurs bat systématiquement la diversification (top_n = 3 ou 4). Le signal est assez fort pour que la concentration l'amplifie, plutôt que de l'amortir.
3. **Sharpe insensible au levier** — les 4 meilleures lignes ont des leviers de 1.0 à 2.0 et **le même Sharpe** (2.016). C'est attendu : le levier scale rendement et volatilité proportionnellement, donc le rendement ajusté du risque ne bouge pas. Optimiser le Sharpe sur le levier est une **illusion** — le levier est un choix de tolérance au risque, pas un paramètre de signal.
4. **Seuil VIX élevé (35 > 25)** — le filtre gagne à être peu actif. À 35, seuls ~6 % des jours sont filtrés ; à 25, ~29 %. Le filtre VIX coûte plus (opportunités manquées) qu'il ne protège, sur cet échantillon.

### Configuration recommandée

```python
# Dans main.py ou le modèle Alpha
LOOKBACK_PERIOD = 90    # momentum 3-mois > 6-mois (décroissance rapide du signal sectoriel)
VIX_THRESHOLD = 35      # filtre minimal — le VIX coûte plus qu'il ne protège ici
LEVERAGE = 1.0          # le Sharpe est insensible au levier ; 1.0 = pas de ruine possible
TOP_N_SECTORS = 2       # concentration — le signal est assez fort pour l'amplifier
```

Sharpe attendu (en échantillon) : **~2.0**, contre **1.65** pour la baseline 126/25/1.5.

### Réserve méthodologique

Le drawdown maximal affiché pour la configuration optimisée (~-6 %) semble bas au regard du rendement cumulé (~3720x) et demande à être **vérifié** : sur un rendement de cet ordre, un drawdown de 6 % est mécaniquement suspect (soit la métrique est calculée sur une base différente de la baseline, soit le levier n'est pas appliqué uniformément dans le grid). Avant tout déploiement, il faut reconcilier l'échelle du drawdown entre la baseline (-120 % à levier 1.5) et le grid (-6 %), et confirmer que le levier est effectivement appliqué. C'est la différence entre un backtest présentable et un backtest investissable.
